# Mirage · official test-suite accuracy (seed 0) — CPU only, no GPU quota

Re-scores the seed-0 dev predictions already in the repo (`results/run2/eval/*/scored_items.jsonl`: base,
GRPO single, GRPO suite) with the **official** Spider test-suite evaluator (github.com/taoyds/test-suite-sql-eval)
and its released test-suite databases. ~1 h.

**Settings:** Accelerator **None**, Internet **On**, input `spider` (Your Datasets). Then **Save Version → Save & Run All (Commit)**.
Result: last cell, or **Output → official/OFFICIAL.md**.

In [ ]:
import os, subprocess, sys, json, glob, re, time, shutil
T0 = time.time()
W = "/kaggle/working"; os.chdir(W)
def sh(cmd, check=True):
    print("$", cmd, flush=True); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    if check and r.returncode: raise RuntimeError(cmd)
    return r
sh("git clone -q --depth 1 https://github.com/SiddharthaRaja/Mirage mirage")
sh("git clone -q --depth 1 https://github.com/taoyds/test-suite-sql-eval tse")
sh(f"{sys.executable} -m pip install -q gdown sqlparse nltk tqdm")
import nltk; ok = [nltk.download(x, quiet=True) for x in ("punkt", "punkt_tab")]; print("nltk data:", ok); assert any(ok), "nltk download failed: is Internet On?"
SPIDER_DIR = "/kaggle/input/datasets/siddhartharajat/spider/spider_data"
assert os.path.exists(f"{SPIDER_DIR}/tables.json"), "attach the spider dataset"

In [ ]:
# --- the official test-suite databases (Google Drive, ~1 GB) ------------------
if not os.path.isdir("tse/database"):
    r = sh("gdown --fuzzy 'https://drive.google.com/file/d/1mkCx2GOFIqNesD4y8TDAO1yX1QZORP5w/view?usp=sharing' -O ts.zip", check=False)
    if not os.path.exists("ts.zip"):
        raise SystemExit("Google Drive download failed (quota/blocked). Download the zip from the test-suite-sql-eval README "
                         "on your laptop, upload it as a private Kaggle dataset, attach it, and point TS_DIR at it.")
    sh("unzip -q ts.zip -d ts_raw")
    cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("ts_raw/**/concert_singer/*.sqlite", recursive=True)]
    assert cands, "unexpected zip layout: " + str(os.listdir("ts_raw"))
    shutil.move(cands[0], "tse/database")
TS_DIR = os.path.abspath("tse/database")
print("test-suite DBs:", len(os.listdir(TS_DIR)), "| concert_singer variants:", len(glob.glob(f"{TS_DIR}/concert_singer/*.sqlite")))

In [ ]:
# --- run the evaluator on each model's greedy predictions --------------------
OUT = f"{W}/official"; os.makedirs(OUT, exist_ok=True)
MODELS = {"base": "start-dev", "GRPO single": "grpo-single-dev", "GRPO suite": "grpo-suite-dev"}
res = {}
for name, d in MODELS.items():
    items = [json.loads(l) for l in open(f"{W}/mirage/results/run2/eval/{d}/scored_items.jsonl")]
    ours = sum(i["ex"] for i in items) / len(items); ours_suite = sum(i["suite_ex"] for i in items) / len(items)
    g, p = f"{OUT}/{d}_gold.txt", f"{OUT}/{d}_pred.txt"
    open(g, "w").write("\n".join(f"{' '.join(i['gold_sql'].split())}\t{i['db_id']}" for i in items) + "\n")
    open(p, "w").write("\n".join(" ".join((i["pred_sql"] or "SELECT 1").split()) for i in items) + "\n")
    t = time.time()
    r = subprocess.run([sys.executable, "evaluation.py", "--gold", g, "--pred", p, "--db", TS_DIR,
                        "--table", f"{SPIDER_DIR}/tables.json", "--etype", "exec"],
                       cwd=f"{W}/tse", text=True, capture_output=True)
    open(f"{OUT}/{d}_stdout.txt", "w").write(r.stdout + "\n--- stderr ---\n" + r.stderr)
    print(f"== {name}: {(time.time()-t)/60:.1f} min, rc={r.returncode}"); print(r.stdout[-1500:]); print(r.stderr[-1500:])
    acc = None
    for line in r.stdout.splitlines():
        if line.strip().lower().startswith("execution"):
            nums = re.findall(r"\d+\.\d+", line)
            if nums: acc = float(nums[-1])
    res[name] = dict(n=len(items), ours_ex=ours, ours_suite_ex=ours_suite, official=acc)
print(json.dumps(res, indent=1))

In [ ]:
lines = ["# Official test-suite execution accuracy · Spider dev · seed 0", "",
         "Same greedy predictions as the dev evaluation; official evaluator (taoyds/test-suite-sql-eval), with values, distinct ignored.", "",
         "| model | EX (single DB) | Mirage suite EX | official test-suite EX |", "|---|---|---|---|"]
for k, v in res.items():
    off = f"{100*v['official']:.1f}%" if v["official"] is not None else "parse failed, see stdout"
    lines.append(f"| {k} | {100*v['ours_ex']:.1f}% | {100*v['ours_suite_ex']:.1f}% | **{off}** |")
lines += ["", f"runtime {(time.time()-T0)/60:.0f} min"]
txt = "\n".join(lines); open(f"{OUT}/OFFICIAL.md", "w").write(txt); print(txt)
if any(v["official"] is None for v in res.values()): raise SystemExit("evaluator output not parsed: read official/*_stdout.txt")